# Step 2 — Disease genes on the interactome, and their overlap

1. Choose the score cutoff: `CUTOFF = 0.1`, a low cutoff that brings as many genes as possible into the analysis, while still removing very weakly associated genes and keeping the computation manageable.
2. Load the interactome (only interactions where both genes have a symbol).
3. For each disease keep the **associated** genes (Open Targets `globalScore` ≥ `CUTOFF`) that are **on the network**.
4. Count shared genes between each pair of diseases and test whether the overlap is bigger than chance (hypergeometric test).

In [1]:
import math
from itertools import combinations
from pathlib import Path

import networkx as nx
import pandas as pd

DISEASE_FILES = {
    "OC":  "./inputs/diseases/ovarian cancer-MONDO_0008170.tsv",
    "HTN": "./inputs/diseases/hypertensive disorder-MONDO_0005044.tsv",
    "SBP": "./inputs/diseases/systolic blood pressure-EFO_0006335.tsv",
}

INTERACTOME_FILE = "./outputs/interactome/interactome_both_symbols.csv"
CUTOFF = 0.1
OUT_DIR = Path(f"./outputs/cutoff_{CUTOFF}")
OUT_DIR.mkdir(parents=True, exist_ok=True)


## Load the interactome

In [2]:
interactome_df = pd.read_csv(INTERACTOME_FILE)

G = nx.from_pandas_edgelist(interactome_df, "symbol_1", "symbol_2")
network_genes = set(G.nodes())

print(f"Genes (nodes):        {G.number_of_nodes()}")
print(f"Interactions (edges): {G.number_of_edges()}")

Genes (nodes):        13270
Interactions (edges): 137863


## Load disease gene lists
Keep genes with `globalScore >= CUTOFF`, then keep only the ones that exist in the network
(genes with no known interactions can't be analysed).

In [3]:
disease_scores = {}
disease_genes = {}
summary = []

for name, path in DISEASE_FILES.items():
    df = pd.read_csv(path, sep="\t", low_memory=False)
    disease_scores[name] = df[["symbol", "globalScore"]]

    strong = set(df.loc[df["globalScore"] >= CUTOFF, "symbol"])
    on_network = strong & network_genes
    disease_genes[name] = on_network

    summary.append({
        "disease": name,
        "all genes in file": len(df),
        f"genes with score >= {CUTOFF}": len(strong),
        "on the network": len(on_network),
        "percentage": len(on_network) / len(strong) * 100,
    })

pd.DataFrame(summary)

,disease,all genes in file,genes with score >= 0.1,on the network,percentage
0,OC,15834,1555,1428,91.832797
1,HTN,10012,2402,1830,76.186511
2,SBP,4884,2877,2158,75.008690


## Gene overlap between diseases

- **shared**: number of genes in both lists
- **Jaccard** = shared / all genes in either list (0 = nothing in common, 1 = identical)
- **overlap coef.** = shared / size of the smaller list
- **expected**: how many shared genes you'd get if both lists were picked at random from the network
- **fold**: shared / expected (> 1 means more overlap than chance)
- **p-value**: hypergeometric test, probability of getting at least this many shared genes by chance

In [4]:
def hypergeom_sf(k, N, K, n):
    """P(X >= k) when drawing n genes out of N, of which K are 'successes'."""
    def log_comb(a, b):
        return math.lgamma(a + 1) - math.lgamma(b + 1) - math.lgamma(a - b + 1)

    denom = log_comb(N, n)
    return sum(
        math.exp(log_comb(K, i) + log_comb(N - K, n - i) - denom)
        for i in range(k, min(K, n) + 1)
    )


N = len(network_genes)  # background: all genes in the network
overlap_rows = []

for a, b in combinations(disease_genes, 2):
    A, B = disease_genes[a], disease_genes[b]
    shared = A & B
    expected = len(A) * len(B) / N
    overlap_rows.append({
        "pair": f"{a} - {b}",
        "size A": len(A),
        "size B": len(B),
        "shared": len(shared),
        "Jaccard": len(shared) / len(A | B),
        "overlap coef.": len(shared) / min(len(A), len(B)),
        "expected": expected,
        "fold": len(shared) / expected,
        "p-value": hypergeom_sf(len(shared), N, len(A), len(B)),
    })

overlap_df = pd.DataFrame(overlap_rows)
overlap_df

,pair,size A,size B,shared,Jaccard,overlap coef.,expected,fold,p-value
0,OC - HTN,1428,1830,383,0.133217,0.268207,196.928410,1.944869,6.813648e-44
1,OC - SBP,1428,2158,343,0.105766,0.240196,232.224868,1.477017,6.647034e-16
2,HTN - SBP,1830,2158,869,0.278615,0.474863,297.599096,2.920036,5.649216e-262


## Which genes are shared?

In [5]:
for a, b in combinations(disease_genes, 2):
    shared = sorted(disease_genes[a] & disease_genes[b])
    print(f"{a} & {b} ({len(shared)} genes):")
    print(", ".join(shared))
    print()

shared_all = sorted(disease_genes["OC"] & disease_genes["HTN"] & disease_genes["SBP"])
print(f"Shared by all three ({len(shared_all)} genes):")
print(", ".join(shared_all))

OC & HTN (383 genes):
ABL1, ABO, ACKR3, ACSL3, ACSL6, ADH1B, ADRB1, ADRB2, ADRB3, AGTR1, AKT1, AKT3, ALDH2, ALK, ANGPT2, APC, APOA1, AR, ASXL1, ASXL2, ATF1, ATP1A1, AXIN2, BABAM1, BCAR1, BCL11B, BCL2L11, BCR, BIRC2, BIRC5, BMP2, BMP5, BMPR1A, BNC2, BRCA1, BRD2, BRD4, BTG1, CACNA1D, CACNA1H, CAMTA1, CAV1, CCDC170, CCDC88C, CCNE1, CD44, CD8A, CDC73, CDH11, CDH17, CDK6, CDKN1B, CDKN2A, CDKN2B, CLDN7, CLTC, CNTNAP2, COL11A1, COL2A1, COL3A1, COXFA4, COXFA4L2, CPS1, CPT1A, CREB3L1, CREBBP, CRTC1, CRTC3, CSF1R, CTLA4, CTNNB1, CUX1, CXCL12, CXCR4, CYP19A1, DCC, DNAJC1, DNMT3A, DOT1L, DPP4, EBF1, EDNRA, EFEMP1, EML4, ENG, EPAS1, EPCAM, EPHA3, EPHB2, EPHB3, ERCC4, ERG, ESR1, EWSR1, EZH2, FAM135B, FANCL, FAT1, FBLN2, FBXO11, FES, FGF5, FGF9, FGFR2, FKBP1A, FLCN, FLNA, FLT1, FLT4, FMN1, FOXM1, FOXO3, FOXP1, FTO, FURIN, GBA1, GDF15, GNAS, GOPC, GPD2, GRIN2A, GSK3B, GTF2I, HDAC4, HDAC7, HDAC9, HGF, HIF1A, HMGA1, HMGA2, HMGB1, HMOX1, HOXA11, HOXA13, HRH1, HRH2, HSPB7, ID1, IFNAR2, IGF1, IGF1R, IGF2BP

## Does the result depend on the cutoff?
Repeat the overlap for several cutoffs. A robust finding should hold across them.
0.2 is the lowest cutoff worth testing: below it, genes can be linked to a disease by weak sources alone (see the cutoff check at the top).

In [6]:
sensitivity_rows = []

for cutoff in [0.2, 0.3, 0.4, 0.5]:
    genes = {
        name: set(s.loc[s["globalScore"] >= cutoff, "symbol"]) & network_genes
        for name, s in disease_scores.items()
    }
    for a, b in combinations(genes, 2):
        A, B = genes[a], genes[b]
        shared = len(A & B)
        expected = len(A) * len(B) / N
        sensitivity_rows.append({
            "cutoff": cutoff,
            "pair": f"{a} - {b}",
            "size A": len(A),
            "size B": len(B),
            "shared": shared,
            "fold": shared / expected,
            "p-value": hypergeom_sf(shared, N, len(A), len(B)),
        })

pd.DataFrame(sensitivity_rows)

,cutoff,pair,size A,size B,shared,fold,p-value
0,0.2,OC - HTN,833,1297,164,2.014326,2.270912e-19
1,0.2,OC - SBP,833,1627,179,1.752635,6.303815e-15
2,0.2,HTN - SBP,1297,1627,623,3.917702,1.265034e-258
3,0.3,OC - HTN,579,726,76,2.399216,6.306261e-13
4,0.3,OC - SBP,579,1022,90,2.018292,6.648799e-11
5,0.3,HTN - SBP,726,1022,384,6.867752,4.274591e-253
6,0.4,OC - HTN,403,308,26,2.779640,2.543029e-06
7,0.4,OC - SBP,403,492,43,2.877857,3.910038e-10
8,0.4,HTN - SBP,308,492,176,15.412311,4.976575e-179
9,0.5,OC - HTN,174,113,1,0.674906,7.763918e-01


## Save the gene lists for the next steps (localization and separation)
One gene symbol per line, same format as `MS.txt` / `PD.txt` in the Menche code.

In [7]:
gene_list_dir = OUT_DIR / "gene_lists"
gene_list_dir.mkdir(parents=True, exist_ok=True)

for name, genes in disease_genes.items():
    path = gene_list_dir / f"{name}.txt"
    with open(path, "w") as f:
        f.write("\n".join(sorted(genes)) + "\n")
    print(f"Saved {len(genes)} genes to {path}")

overlap_df.to_csv(OUT_DIR / "gene_overlap.csv", index=False)
print(f"Saved {OUT_DIR / 'gene_overlap.csv'}")

Saved 1428 genes to outputs/cutoff_0.1/gene_lists/OC.txt
Saved 1830 genes to outputs/cutoff_0.1/gene_lists/HTN.txt
Saved 2158 genes to outputs/cutoff_0.1/gene_lists/SBP.txt
Saved outputs/cutoff_0.1/gene_overlap.csv


---
# Network analysis (Menche et al. 2015)

Gene overlap only counts genes that are *identical*. The network analysis also uses genes that are **close to each other** on the interactome.
We reproduce the two measures from the Menche paper (same definitions as `localization_py3.py` and `separation_py3.py`):

**1. Localization: do a disease's genes cluster together on the network?**
- **LCC (S)**: size of the largest connected component, i.e. the biggest group of disease genes that are directly linked to each other.
- **d_s**: for every disease gene, the distance (number of steps) to the closest *other* gene of the same disease, averaged.
- Both are compared with random gene sets of the same size, repeated `N_SIMS` times.
  **z-score > 1.6** (p < 0.05) for LCC means the disease genes form a real cluster (a *disease module*).

**2. Separation: how far apart are two diseases on the network?**

$$s_{AB} = \langle d_{AB} \rangle - \frac{\langle d_{AA} \rangle + \langle d_{BB} \rangle}{2}$$

- **s_AB < 0**: the two disease modules **overlap** (shared biology is likely).
- **s_AB > 0**: the modules are **separated**.
- A negative z-score means the diseases are closer than two random gene sets of the same sizes.

The distance between every pair of genes is computed once and stored in a matrix. Every gene set (real or random) then just
looks its distances up, which gives **exactly the same numbers** as Menche's pair-by-pair code but is far faster
(needed for the random repeats).

In [8]:
import multiprocessing as mp
import os
import random
from collections import deque

import numpy as np

N_SIMS = 10000            # number of random gene sets (Menche used 1000 for separation)
RANDOM_MODEL = "uniform"  # "uniform" = Menche's full randomization; "degree" = random genes with similar degree
SEED = 42                 # makes the random results reproducible
N_WORKERS = os.cpu_count()  # processes used for the random repeats

net_genes = sorted(network_genes)

## Distance between every pair of genes
Computed once and reused for all the random repeats. Takes a minute or two and about 180 MB of memory.

In [9]:
gene_index = {g: i for i, g in enumerate(net_genes)}
neighbours = [[gene_index[v] for v in G[g]] for g in net_genes]
UNREACHABLE = 255  # marks gene pairs with no path between them (real distances are far smaller)


def distances_from(source):
    """Breadth-first search: distance from one gene to every gene in the network."""
    row = np.full(len(net_genes), UNREACHABLE, dtype=np.uint8)
    dist = {source: 0}
    queue = deque([source])
    while queue:
        u = queue.popleft()
        for v in neighbours[u]:
            if v not in dist:
                dist[v] = dist[u] + 1
                queue.append(v)
    row[list(dist)] = list(dist.values())
    return row


def _distance_rows(sources):
    """Distance rows for a block of source genes (one task for a worker process)."""
    return sources.start, np.vstack([distances_from(s) for s in sources])


# D[i, j] = number of steps between gene i and gene j
# The rows are split into blocks and computed on N_WORKERS processes.
n_genes = len(net_genes)
block = 200
D = np.empty((n_genes, n_genes), dtype=np.uint8)
blocks = [range(i, min(i + block, n_genes)) for i in range(0, n_genes, block)]
# "fork" (Linux only): workers inherit `neighbours` without pickling it
with mp.get_context("fork").Pool(N_WORKERS) as pool:
    for done, (start, rows) in enumerate(pool.imap_unordered(_distance_rows, blocks), 1):
        D[start:start + len(rows)] = rows
        print(f"  distances: {done}/{len(blocks)} blocks", end="\r")

print(f"Distance matrix: {D.shape[0]} x {D.shape[1]} genes, {D.nbytes / 1e6:.0f} MB")
print(f"Longest distance: {D[D < UNREACHABLE].max()} steps")

Distance matrix: 13270 x 13270 genes, 176 MB
Longest distance: 13 steps


## Helper functions

In [10]:
def lcc_size(genes):
    """Size of the largest connected component formed by `genes` (Menche: S)."""
    sub = G.subgraph(genes)
    if sub.number_of_nodes() == 0:
        return 0
    return len(max(nx.connected_components(sub), key=len))


def mean_within_distance(genes):
    """<d_s> = <d_AA>: mean distance of each gene to the closest OTHER gene of the same set."""
    a = [gene_index[g] for g in genes]
    sub = D[np.ix_(a, a)].copy()
    np.fill_diagonal(sub, UNREACHABLE)  # a gene is not its own neighbour
    nearest = sub.min(axis=1)
    return float(nearest[nearest < UNREACHABLE].mean())


def mean_between_distance(genes_a, genes_b):
    """<d_AB>: distance of every A gene to the closest B gene, and of every B gene to the closest A gene (shared genes count as 0)."""
    a = [gene_index[g] for g in genes_a]
    b = [gene_index[g] for g in genes_b]
    sub = D[np.ix_(a, b)]
    nearest = np.concatenate([sub.min(axis=1), sub.min(axis=0)])  # A -> B, then B -> A
    return float(nearest[nearest < UNREACHABLE].mean())


def separation(genes_a, genes_b):
    d_a, d_b = mean_within_distance(genes_a), mean_within_distance(genes_b)
    d_ab = mean_between_distance(genes_a, genes_b)
    return d_ab - (d_a + d_b) / 2, d_a, d_b, d_ab


# ---- random gene sets -------------------------------------------------------
# "degree" model: group genes into bins of similar degree (number of links, at least 100 genes per bin)
# and replace every disease gene with a random gene from its own bin. This controls for disease genes
# being well-studied "hub" genes with many links.
degree_bins = []
_current = []
for gene in sorted(net_genes, key=G.degree):
    _current.append(gene)
    if len(_current) >= 100 and G.degree(gene) != G.degree(_current[0]):
        degree_bins.append(_current)
        _current = []
degree_bins[-1].extend(_current)
bin_of = {g: i for i, b in enumerate(degree_bins) for g in b}


def random_like(genes, rng):
    """A random gene set of the same size (and, for the 'degree' model, similar degrees)."""
    if RANDOM_MODEL == "uniform":
        return set(rng.sample(net_genes, len(genes)))
    picked = set()
    for g in genes:
        choice = rng.choice(degree_bins[bin_of[g]])
        while choice in picked:
            choice = rng.choice(degree_bins[bin_of[g]])
        picked.add(choice)
    return picked


def z_and_p(observed, random_values, larger_is_significant):
    """z-score and empirical one-sided p-value of the observed value vs the random values."""
    random_values = np.asarray(random_values)
    z = (observed - random_values.mean()) / random_values.std()
    if larger_is_significant:
        p = (np.sum(random_values >= observed) + 1) / (len(random_values) + 1)
    else:
        p = (np.sum(random_values <= observed) + 1) / (len(random_values) + 1)
    return z, p

In [11]:
# ---- random repeats on several processes ---------------------------------------
# The repeats are split into N_CHUNKS chunks, each with its own seed, so the results
# are the same whatever N_WORKERS is.
N_CHUNKS = 100


def _localization_chunk(task):
    genes, seed, n = task
    rng = random.Random(seed)
    out = []
    for _ in range(n):
        r = random_like(genes, rng)
        out.append((lcc_size(r), mean_within_distance(r)))
    return out


def _separation_chunk(task):
    genes_a, genes_b, seed, n = task
    rng = random.Random(seed)
    return [separation(random_like(genes_a, rng), random_like(genes_b, rng))[0] for _ in range(n)]


def run_random(worker, args, label):
    """Run N_SIMS random repeats of `worker` on N_WORKERS processes; returns one result per repeat."""
    sizes = [N_SIMS // N_CHUNKS + (k < N_SIMS % N_CHUNKS) for k in range(N_CHUNKS)]
    tasks = [(*args, f"{SEED}-{label}-{k}", n) for k, n in enumerate(sizes) if n]
    results = []
    # "fork" (Linux only): workers inherit D, G and the functions above without copying or pickling them
    with mp.get_context("fork").Pool(N_WORKERS) as pool:
        for chunk in pool.imap(worker, tasks):
            results.extend(chunk)
            print(f"  {label}: {len(results)}/{N_SIMS} random repeats", end="\r")
    return results

## 1. Localization of each disease (LCC and d_s)
Takes about 10 seconds.

In [12]:
loc_rows = []

for name, genes in disease_genes.items():
    S = lcc_size(genes)
    d_s = mean_within_distance(genes)

    rand_S, rand_d = zip(*run_random(_localization_chunk, (genes,), name))

    z_S, p_S = z_and_p(S, rand_S, larger_is_significant=True)
    z_d, p_d = z_and_p(d_s, rand_d, larger_is_significant=False)
    loc_rows.append({
        "disease": name,
        "genes on network": len(genes),
        "LCC size S": S,
        "LCC fraction": S / len(genes),
        "random LCC (mean)": np.mean(rand_S),
        "LCC z-score": z_S,
        "LCC p-value": p_S,
        "d_s": d_s,
        "random d_s (mean)": np.mean(rand_d),
        "d_s z-score": z_d,
        "d_s p-value": p_d,
    })
    print(f"{name}: LCC = {S} (random {np.mean(rand_S):.1f}), z = {z_S:.1f}              ")

localization_df = pd.DataFrame(loc_rows)
localization_df

OC: LCC = 1283 (random 705.0), z = 14.5              
HTN: LCC = 1292 (random 1025.9), z = 6.3              
SBP: LCC = 1533 (random 1302.8), z = 5.1              


,disease,genes on network,LCC size S,LCC fraction,random LCC (mean),LCC z-score,LCC p-value,d_s,random d_s (mean),d_s z-score,d_s p-value
0,OC,1428,1283,0.898459,704.9748,14.476220,0.0001,1.093903,1.471888,-15.748191,0.0001
1,HTN,1830,1292,0.706011,1025.9381,6.257281,0.0001,1.278904,1.410270,-6.426463,0.0001
2,SBP,2158,1533,0.710380,1302.7928,5.132904,0.0001,1.277545,1.369932,-4.993207,0.0001


**How to read it:** LCC z-score > 1.6 means the disease genes cluster together more than random genes do,
so the disease forms a network module. Only then does it make sense to measure separation between diseases.
A negative d_s z-score means the disease genes are closer to each other than random genes are.

## 2. Separation between each pair of diseases (s_AB)
Takes a few seconds per pair with `N_SIMS = 1000`.

In [13]:
sep_rows = []

for a, b in combinations(disease_genes, 2):
    A, B = disease_genes[a], disease_genes[b]
    s_ab, d_a, d_b, d_ab = separation(A, B)

    rand_s = run_random(_separation_chunk, (A, B), f"{a} - {b}")

    z, p = z_and_p(s_ab, rand_s, larger_is_significant=False)
    sep_rows.append({
        "pair": f"{a} - {b}",
        "d_AA": d_a,
        "d_BB": d_b,
        "d_AB": d_ab,
        "s_AB": s_ab,
        "random s_AB (mean)": np.mean(rand_s),
        "z-score": z,
        "p-value (closer than random)": p,
        "modules": "overlapping" if s_ab < 0 else "separated",
    })
    print(f"{a} - {b}: s_AB = {s_ab:.3f}, z = {z:.1f}              ")

separation_df = pd.DataFrame(sep_rows)
separation_df

OC - HTN: s_AB = -0.233, z = -3.9              
OC - SBP: s_AB = -0.178, z = -0.2              
HTN - SBP: s_AB = -0.536, z = -23.2              


,pair,d_AA,d_BB,d_AB,s_AB,random s_AB (mean),z-score,p-value (closer than random),modules
0,OC - HTN,1.093903,1.278904,0.953538,-0.232865,-0.171175,-3.897645,0.000100,overlapping
1,OC - SBP,1.093903,1.277545,1.007550,-0.178174,-0.174423,-0.249934,0.407059,overlapping
2,HTN - SBP,1.278904,1.277545,0.742341,-0.535884,-0.206726,-23.233096,0.000100,overlapping


**How to read it:**
- **s_AB < 0**: the disease modules overlap on the network.
- **z-score < -1.6** (p < 0.05): the two diseases are significantly closer than random gene sets of the same sizes.
- HTN – SBP should come out strongly overlapping: that is the **positive control**.
- Shared genes count as distance 0, so they pull s_AB down. With the "uniform" model the random sets share almost no genes,
  so part of a negative z-score simply reflects the shared genes (see Menche SM section 2.2.1).

In [14]:
localization_df.to_csv(OUT_DIR / f"localization_{RANDOM_MODEL}.csv", index=False)
separation_df.to_csv(OUT_DIR / f"separation_{RANDOM_MODEL}.csv", index=False)
print(f"Saved {OUT_DIR / f'localization_{RANDOM_MODEL}.csv'}")
print(f"Saved {OUT_DIR / f'separation_{RANDOM_MODEL}.csv'}")

Saved outputs/cutoff_0.1/localization_uniform.csv
Saved outputs/cutoff_0.1/separation_uniform.csv
